# Auxiliary viral genes across global soil and human-gut viral communities

Reconcile `annotate` and `de-novo` AVG calls across the MetaVR soil and human-gut collections and write row-level tables for soil_gut_avgs_figures.Rmd, which does all counting, testing, and multiple-testing correction. Propagated labels come from functional_propagation.ipynb.

## 1. Setup

Imports and thread configuration.

In [1]:
import os
os.environ["POLARS_MAX_THREADS"] = "64"
import json
import time
import gc
from pathlib import Path
import numpy as np
import polars as pl
import tables as tb
import faiss
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(90)

polars.config.Config

Paths and constants.

In [2]:
SEED = 20260722
N_THREADS = 64
faiss.omp_set_num_threads(N_THREADS)
CAT3 = ["metabolic", "physiological", "regulatory"]
VIRAL_OK = ["medium", "high", "very high"]
DATASETS = ["soil", "gut"]

REPO = Path("/storage2/scratch/kosmopoulos/software/CheckAMG")
FILES = REPO / "CheckAMG" / "files"
MAIN = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript")
TABLES = REPO / "notebooks" / "tables" / "soil_gut_avgs"
TABLES.mkdir(parents=True, exist_ok=True)
PROP = REPO / "notebooks" / "tables" / "propagation"

def den(ds):
    return MAIN / f"CheckAMG_denovo_v1.1_MetaVR{ds}"
def anno(ds):
    return MAIN / f"CheckAMG_annotate_v1.1_MetaVR{ds}" / "results"
def agg(ds):
    return MAIN / f"CheckAMG_aggregate_v1.1_MetaVR{ds}" / "aggregated_results_detailed.parquet"

METAVR = Path("/storage2/databases/metaVR")
IPHOP = METAVR / "MetaVR_iPHoP_results.tsv"
SRC_META = METAVR / "Source_dataset_metadata.tsv"
def uvm_path(ds):
    return REPO / "notebooks" / "data" / f"metavr_uvigs_{ds}.csv.gz"

DENOVO_DB = REPO / "notebooks" / "CheckAMG_denovo_db_v1.1_20260714"
DB_STAMP = DENOVO_DB.name.replace("CheckAMG_denovo_db_", "")
OUT = MAIN / "soil_gut_avgs" / DB_STAMP
OUT.mkdir(parents=True, exist_ok=True)

def fresh(cache, *inputs):
    cache = Path(cache)
    if not cache.exists():
        return False
    ct = cache.stat().st_mtime
    return all(Path(i).exists() and ct >= Path(i).stat().st_mtime for i in inputs)

def save_tsv(df, name):
    p = TABLES / name
    df.write_csv(p, separator="\t")
    print(f"wrote {p.name} ({df.height:,}x{df.width})")
    return p

Load the canonical assignments and check the schema version and the `label_source` vocabulary, so a stale table fails immediately.

In [3]:
SCHEMA_VERSION = "propagation-1.2"
manifest = json.loads((PROP / "manifest.json").read_text())
assert manifest["schema_version"] == SCHEMA_VERSION, \
    f"propagation schema {manifest['schema_version']} != expected {SCHEMA_VERSION}; rerun functional_propagation.ipynb"
# Renamed back to sequence_similarity_invisible when the master and category tables are written
assign = pl.read_parquet(PROP / "protein_assignments.parquet").rename({"sequence_similarity_invisible": "homology_invisible"})
_TIER_COLS = ["label_source", "tier1_label", "tier1_level", "tier2_label", "tier2_level", "cluster_reachable"]
_absent = [c for c in _TIER_COLS if c not in assign.columns]
assert not _absent, f"tiered propagation columns absent: {_absent}; rerun functional_propagation.ipynb"
_SRC_VOCAB = {"cluster", "embedding", "none"}
assert _SRC_VOCAB <= set(manifest["tier_definitions"]), "manifest does not document the three label sources"
assert set(assign["label_source"].unique()) <= _SRC_VOCAB, "label_source carries an undocumented value"
assert assign["label_source"].null_count() == 0, "label_source must be populated for every protein"
assert assign["final_level"].null_count() == 0, "final_level must be populated for every protein"
print(f"assignments: {assign.height:,} x {assign.width}  (schema {manifest['schema_version']}, "
      f"target {manifest['primary_target']:.0%}, generated {manifest['generated']})")
print(assign.group_by("final_level").agg(pl.len().alias("n")).sort("n", descending=True))
print(assign.group_by("label_source").agg(pl.len().alias("n")).sort("n", descending=True))
print(assign.group_by(["final_level", "label_source"]).agg(pl.len().alias("n"))
      .sort(["final_level", "n"], descending=[False, True]))

assignments: 541,199 x 43  (schema propagation-1.2, target 90%, generated 2026-09-23T22:03:58)
shape: (4, 2)
┌─────────────┬────────┐
│ final_level ┆ n      │
│ ---         ┆ ---    │
│ str         ┆ u32    │
╞═════════════╪════════╡
│ specific    ┆ 374304 │
│ L1          ┆ 80216  │
│ unassigned  ┆ 77301  │
│ category    ┆ 9378   │
└─────────────┴────────┘
shape: (3, 2)
┌──────────────┬────────┐
│ label_source ┆ n      │
│ ---          ┆ ---    │
│ str          ┆ u32    │
╞══════════════╪════════╡
│ cluster      ┆ 336437 │
│ embedding    ┆ 127461 │
│ none         ┆ 77301  │
└──────────────┴────────┘
shape: (7, 3)
┌─────────────┬──────────────┬────────┐
│ final_level ┆ label_source ┆ n      │
│ ---         ┆ ---          ┆ ---    │
│ str         ┆ str          ┆ u32    │
╞═════════════╪══════════════╪════════╡
│ L1          ┆ cluster      ┆ 66493  │
│ L1          ┆ embedding    ┆ 13723  │
│ category    ┆ embedding    ┆ 9110   │
│ category    ┆ cluster      ┆ 268    │
│ specific    ┆ clu

## 2. Reconciled AVG calls

Annotate columns come from each dataset's aggregate and the de-novo probabilities come from `predictions.tsv`. An **annotate AVG** is a metabolic, physiological, or regulatory classification with medium or higher annotate viral confidence. A **de-novo AVG** has Final-AVG and viral probability both at medium confidence or higher. A **de-novo-only AVG** is a de-novo AVG that annotate did not call an AVG, and the **sequence-similarity invisible** subset additionally carries no annotate HMM hit at all.

These flags are recomputed here only to obtain per-genome totals. The AVG membership itself is identical to what the propagation notebook derived, and the cell below asserts that.

Reconcile the two modules across both datasets.

In [4]:
pst_thr = json.loads((FILES / "pst_thresholds.json").read_text())
VMED, AMED = pst_thr["Viral"]["medium"], pst_thr["AVG"]["medium"]
ANN = ["Protein", "Genome", "Classification (annotate)", "Viral Confidence Level (annotate)", "Function (annotate)"]
comb = pl.concat([
    pl.read_parquet(agg(ds), columns=ANN)
      .join(pl.read_csv(den(ds) / "predictions.tsv", separator="\t",
                        columns=["Protein", "Viral prob", "Final AVG prob"]), on="Protein", how="left")
      .with_columns(pl.lit(ds).alias("dataset")) for ds in DATASETS])
comb = comb.with_columns(
    (pl.col("Classification (annotate)").is_in(CAT3)
     & pl.col("Viral Confidence Level (annotate)").is_in(VIRAL_OK)).alias("annotate_avg"),
    ((pl.col("Final AVG prob") >= AMED) & (pl.col("Viral prob") >= VMED)).alias("denovo_avg"))
comb = comb.with_columns(
    (pl.col("denovo_avg") & ~pl.col("annotate_avg")
     & (pl.col("Function (annotate)").is_null() | (pl.col("Function (annotate)") == ""))
     ).alias("homology_invisible"))
print("proteins:", f"{comb.height:,}")
print("annotate AVGs (medium+):", dict(comb.filter(pl.col("annotate_avg")).group_by("dataset").len().sort("dataset").iter_rows()))
print("de-novo AVGs:", f"{int(comb['denovo_avg'].sum()):,}")
donly = comb.filter(pl.col("denovo_avg") & ~pl.col("annotate_avg"))
print(f"de-novo-only: {donly.height:,} | sequence-similarity invisible: {int(donly['homology_invisible'].sum()):,}")
avg_here = comb.filter(pl.col("annotate_avg") | pl.col("denovo_avg"))
assert avg_here.height == assign.height, f"AVG set {avg_here.height:,} != assignments {assign.height:,}"
print(f"AVG membership matches the propagation table exactly ({avg_here.height:,})")

proteins: 23,375,805
annotate AVGs (medium+): {'gut': 177614, 'soil': 174200}
de-novo AVGs: 484,165
de-novo-only: 189,385 | sequence-similarity invisible: 109,107
AVG membership matches the propagation table exactly (541,199)


## 3. Per-protein AVG master table

Every protein called an AVG by either module, with the reconciled flags, the propagated label and its depth, the AVG PST-embedding cluster, and per-genome ecosystem and predicted-host metadata.

The downstream contract is `function_label = coalesce(annotate_function, final_label)` paired with `function_level`, which is `annotate` for HMM-derived functions and otherwise the propagation depth. No cell may use `function_label` without also respecting `function_level`.

`function_source` is `annotate` for HMM-derived functions and otherwise the tier that supplied the label (`cluster`, `embedding`, or `none`).

Attach genome metadata and build the master.

In [5]:
iph = (pl.read_csv(IPHOP, separator="\t", infer_schema_length=0)
         .with_columns(pl.col("confidence_score").cast(pl.Float64))
         .sort(["confidence_score", "uvig"], descending=[True, False]).unique("uvig", keep="first"))
src = (pl.read_csv(SRC_META, separator="\t", infer_schema_length=0)
         .select([pl.col("IMG Genome ID").alias("taxon_oid"),
                  pl.col("Ecosystem Subtype").alias("ecosystem_subtype")]).unique("taxon_oid"))
uvm = pl.concat([pl.read_csv(uvm_path(ds), infer_schema_length=0).select(["uvig", "taxon_oid"])
                 for ds in DATASETS]).unique("uvig")
gmeta = (assign.select("protein_id").join(comb.select(["Protein", "Genome"]),
                                          left_on="protein_id", right_on="Protein", how="left")
         .select("Genome").unique()
         .with_columns(pl.col("Genome").str.split("|").list.first().alias("uvig"))
         .join(iph.select(["uvig", "host_taxon"]), on="uvig", how="left")
         .join(uvm, on="uvig", how="left").join(src, on="taxon_oid", how="left")
         .with_columns(pl.when(pl.col("ecosystem_subtype").is_in(["Unclassified", "unclassified"]))
                         .then(None).otherwise(pl.col("ecosystem_subtype")).alias("ecosystem_subtype"))
         .select(["Genome", "ecosystem_subtype", pl.col("host_taxon").alias("host_taxonomy")]))
_HAS_ANN = pl.col("annotate_function").is_not_null() & (pl.col("annotate_function") != "")
mast = (assign.rename({"protein_id": "Protein", "dataset": "ecosystem"})
        .join(comb.select(["Protein", "Genome", "Classification (annotate)",
                           "Viral Confidence Level (annotate)"]), on="Protein", how="left")
        .rename({"Classification (annotate)": "annotate_classification",
                 "Viral Confidence Level (annotate)": "annotate_viral_confidence"})
        .with_columns(pl.col("module").is_in(["annotate", "both"]).alias("annotate_avg"),
                      pl.col("module").is_in(["de_novo", "both"]).alias("denovo_avg"))
        .with_columns(pl.coalesce(["annotate_function", "final_label"]).alias("function_label"),
                      pl.when(_HAS_ANN).then(pl.lit("annotate"))
                        .otherwise(pl.col("final_level")).alias("function_level"),
                      pl.when(_HAS_ANN).then(pl.lit("annotate"))
                        .otherwise(pl.col("label_source")).alias("function_source"))
        .join(gmeta, on="Genome", how="left"))
print("master:", mast.shape)
print(mast.group_by("function_level").agg(pl.len().alias("n")).sort("n", descending=True))
print(mast.group_by("function_source").agg(pl.len().alias("n")).sort("n", descending=True))

master: (541199, 53)
shape: (5, 2)
┌────────────────┬────────┐
│ function_level ┆ n      │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ annotate       ┆ 432092 │
│ specific       ┆ 45392  │
│ unassigned     ┆ 37957  │
│ L1             ┆ 20651  │
│ category       ┆ 5107   │
└────────────────┴────────┘
shape: (4, 2)
┌─────────────────┬────────┐
│ function_source ┆ n      │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ annotate        ┆ 432092 │
│ embedding       ┆ 44606  │
│ none            ┆ 37957  │
│ cluster         ┆ 26544  │
└─────────────────┴────────┘


## 4. AVG embedding clustering and biome mixing

Every AVG with a curated `annotate` category and every sequence-similarity invisible AVG, including those without a propagated label, embedded with UMAP and clustered with HDBSCAN. For each protein, the fraction of its 10 nearest AVG neighbors sharing its ecosystem and its category is recorded.

**Parameters.** UMAP `n_neighbors` 30, `min_dist` 0.1, metric cosine, on L2-normalized CheckAMG-PST embeddings. HDBSCAN `min_cluster_size` 300, `min_samples` 30, on the two-dimensional UMAP coordinates. Nearest-neighbor mixing uses a FAISS HNSW inner-product index, `M` 32, `efConstruction` 200, `efSearch` 64, over the 10 nearest neighbors excluding self. Cluster biome skew bands are soil-specific at `soil_frac` at least 0.85, soil-leaning 0.65 to 0.85, mixed 0.35 to 0.65, gut-leaning 0.15 to 0.35, and gut-specific at most 0.15.

Extract embeddings and run a seeded UMAP and HDBSCAN. The results are cached to save runtime and recomputed when the embedding stores change. This stays in Python because it reads the HDF5 embedding stores directly and depends on umap-learn, HDBSCAN, and FAISS.

In [6]:
import umap
from sklearn.cluster import HDBSCAN
UMAP_NPY, UMAP_KEYS, HDB_NPY = OUT / "avg_umap.npy", OUT / "avg_umap_keys.parquet", OUT / "avg_hdbscan.npy"
real = mast.filter((pl.col("annotate_avg") & pl.col("annotate_classification").is_in(CAT3))
                   | pl.col("homology_invisible"))
_umap_inputs = [den("soil") / "combined_proteins.filtered.PST-EMBED.h5",
                den("gut") / "combined_proteins.filtered.PST-EMBED.h5"]
if all(fresh(p, *_umap_inputs) for p in [UMAP_NPY, UMAP_KEYS, HDB_NPY]):
    U = np.load(UMAP_NPY)
    ak = pl.read_parquet(UMAP_KEYS)
    acl = np.load(HDB_NPY)
    print("loaded cached UMAP:", U.shape)
else:
    t0 = time.time()
    emb_c, key_c = [], []
    for ds in DATASETS:
        pred = pl.read_csv(den(ds) / "predictions.tsv", separator="\t", columns=["Protein"]).with_row_index("row")
        want = pred.join(real.filter(pl.col("ecosystem") == ds).select("Protein"), on="Protein", how="inner").sort("row")
        m = np.zeros(pred.height, bool)
        m[want["row"].to_numpy()] = True
        buf = []
        with tb.open_file(str(den(ds) / "combined_proteins.filtered.PST-EMBED.h5")) as fp:
            a = fp.root.ctx_ptn
            n = a.shape[0]
            for s in range(0, n, 500_000):
                e = min(s + 500_000, n)
                mm = m[s:e]
                if mm.any():
                    buf.append(a[s:e][mm])
        emb_c.append(np.concatenate(buf).astype(np.float32))
        key_c.append(want.select(["Protein", pl.lit(ds).alias("ecosystem")]))
    E = np.concatenate(emb_c)
    ak = pl.concat(key_c)
    En = E / np.clip(np.linalg.norm(E, axis=1, keepdims=True), 1e-8, None)
    U = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine", random_state=SEED).fit_transform(En)
    acl = HDBSCAN(min_cluster_size=300, min_samples=30, n_jobs=N_THREADS).fit_predict(U)
    np.save(UMAP_NPY, U)
    ak.write_parquet(UMAP_KEYS)
    np.save(HDB_NPY, acl)
    np.save(OUT / "avg_umap_En.npy", En)
    print(f"UMAP and HDBSCAN in {time.time()-t0:.0f}s:", U.shape)
ak = (ak.with_columns(pl.Series("cluster", acl), pl.Series("umap1", U[:, 0].astype(np.float32)),
                      pl.Series("umap2", U[:, 1].astype(np.float32)))
        .join(mast.select(["Protein", "Genome", "annotate_classification", "raw_category",
                           "function_label", "function_level", "function_source",
                           "raw_L1", "host_taxonomy"]),
              on="Protein", how="left")
        .with_columns(pl.coalesce(["annotate_classification", "raw_category"]).alias("category")))
print(f"real-AVG proteins embedded: {ak.height:,}")
print(ak.group_by("function_level").agg(pl.len().alias("n")).sort("n", descending=True))

loaded cached UMAP: (460729, 2)
real-AVG proteins embedded: 460,729
shape: (5, 2)
┌────────────────┬────────┐
│ function_level ┆ n      │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ annotate       ┆ 351622 │
│ specific       ┆ 45392  │
│ unassigned     ┆ 37957  │
│ L1             ┆ 20651  │
│ category       ┆ 5107   │
└────────────────┴────────┘


Nearest-neighbor biome and category mixing, per protein.

The neighbor search is a FAISS HNSW query over the normalized embeddings, so it stays here. What is emitted is the per-protein fraction of its 10 nearest AVG neighbors sharing its ecosystem and its category, not the mean of those fractions. The Rmd takes the means and the baselines.

In [7]:
En = np.load(OUT / "avg_umap_En.npy")
ix = faiss.IndexHNSWFlat(En.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
ix.hnsw.efConstruction = 200
# Single-threaded insertion makes the HNSW graph, and so the neighbor sets, reproducible
faiss.omp_set_num_threads(1)
ix.add(np.ascontiguousarray(En))
faiss.omp_set_num_threads(N_THREADS)
ix.hnsw.efSearch = 64
_, nn = ix.search(np.ascontiguousarray(En), 11)
nn = nn[:, 1:]
eco = ak["ecosystem"].to_numpy()
ca = ak["category"].to_numpy()
ak = ak.with_columns(
    pl.Series("frac_nn_same_ecosystem", (eco[nn] == eco[:, None]).mean(axis=1).astype(np.float32)),
    pl.Series("frac_nn_same_category", (ca[nn] == ca[:, None]).mean(axis=1).astype(np.float32)))
mast = mast.join(ak.select(["Protein", pl.col("cluster").alias("protein_emb_cluster")]),
                 on="Protein", how="left")
print(f"per-protein neighbor mixing computed for {ak.height:,} proteins")

per-protein neighbor mixing computed for 460,729 proteins


## 5. Row-level output tables

Four tables carry everything soil_gut_avgs_figures.Rmd needs. Each is at the granularity of an analysis unit, and each carries the identifiers needed to join back to the others.

| table | grain | what it supports |
|---|---|---|
| `avg_master_per_protein` | one row per AVG protein | assignment depth, biome comparisons, annotation status, function-level enrichment |
| `avg_protein_categories` | one row per protein per (L1, L2) membership | biogeochemical category analyses at both levels, multi-membership accounting |
| `avg_genome_universe` | one row per screened genome | denominators for every genome-level test, subtype and host grouping |
| `avg_umap_points` | one row per embedded AVG protein | embedding figure, cluster summaries, biome skew |

A protein appears in `avg_protein_categories` once per category it belongs to. That is deliberate: the curated tables let one function sit in several biogeochemical categories, and collapsing to a single representative parent would report an arbitrary choice as a fact.

Annotation-status bucket for the de-novo-only AVGs, per protein.

This is a per-protein classification, not a summary, so it belongs on the master rather than in a counted table. It records why homology missed each de-novo-only call: no annotation at all, or an annotation that exists but does not imply an auxiliary function.

In [8]:
_mob = set(pl.read_csv(FILES / "mobile_genes.tsv", separator="\t")["id"].to_list())
_hall = set(pl.read_csv(FILES / "viral_hallmark_genes.tsv", separator="\t")
            .filter(pl.col("db").is_in(["KEGG", "Pfam"]))["id"].to_list())
_IDC = ("KEGG_hmm_id", "Pfam_hmm_id", "PHROG_hmm_id", "dbCAN_hmm_id",
        "FOAM_hmm_id", "CAMPER_hmm_id", "METABOLIC_hmm_id")
_AUXC = ("KEGG_hmm_id", "Pfam_hmm_id", "dbCAN_hmm_id", "FOAM_hmm_id", "CAMPER_hmm_id", "METABOLIC_hmm_id")
_ga = pl.concat([pl.read_parquet(anno(ds) / "gene_annotations.parquet",
                                 columns=["Protein", "top_hit_db"] + list(_IDC))
                 for ds in DATASETS]).unique("Protein")
_do = (mast.filter(pl.col("denovo_avg") & ~pl.col("annotate_avg"))
       .select(["Protein", "homology_invisible", "annotate_classification"])
       .join(_ga, on="Protein", how="left"))
def _bucket(r):
    if r["homology_invisible"]:
        return "unannotated"
    _i = [str(r[k]) for k in _IDC if r[k] is not None]
    if any(x in _mob for x in _i):
        return "annotated: mobile element"
    if any(x in _hall for x in _i):
        return "annotated: viral hallmark (KEGG/Pfam)"
    if r["top_hit_db"] == "PHROG":
        return "annotated: PHROG phage gene"
    if r["annotate_classification"] in (None, "unclassified") and any(r[k] is not None for k in _AUXC):
        return "annotated: auxiliary-ref hit, still unclassified"
    return "annotated: other database"
_buck = pl.DataFrame({"Protein": _do["Protein"],
                      "q1_annotation_bucket": [_bucket(r) for r in _do.iter_rows(named=True)]})
mast = mast.join(_buck, on="Protein", how="left")
mast = mast.select([
    'Protein',
    'Genome',

    'ecosystem',
    'ecosystem_subtype',

    'annotate_classification',
    'annotate_function',
    'annotate_viral_confidence',
    'annotate_avg',
    'denovo_avg',
    'function_label',
    'function_level',
    'function_source',
    'homology_invisible',

    'host_taxonomy',

    'module',
    'nn_ref_id',
    'd1',
    'd2',
    'margin',
    'ratio',
    'vote_purity_k5',
    'vote_purity_k20',
    'vote_purity_k50',
    'density_norm_d1',
    'hub_count',
    'raw_category',
    'raw_L1',
    'raw_specific',
    'p_category',
    'p_L1',
    'p_specific',
    'final_label',
    'final_level',
    'label_source',
    'tier1_label',
    'tier1_level',
    'tier2_label',
    'tier2_level',
    'cluster_reachable',
    'max_ident_to_train',
    'precision_target',
    'label_80',
    'level_80',
    'source_80',
    'label_70',
    'level_70',
    'source_70',
    'q1_annotation_bucket',

])
print(f"de-novo-only AVGs bucketed: {_buck.height:,}")
print(_buck.group_by("q1_annotation_bucket").agg(pl.len().alias("n")).sort("n", descending=True))

de-novo-only AVGs bucketed: 189,385
shape: (6, 2)
┌──────────────────────────────────────────────────┬────────┐
│ q1_annotation_bucket                             ┆ n      │
│ ---                                              ┆ ---    │
│ str                                              ┆ u32    │
╞══════════════════════════════════════════════════╪════════╡
│ unannotated                                      ┆ 109107 │
│ annotated: PHROG phage gene                      ┆ 29682  │
│ annotated: auxiliary-ref hit, still unclassified ┆ 25594  │
│ annotated: other database                        ┆ 20340  │
│ annotated: mobile element                        ┆ 4073   │
│ annotated: viral hallmark (KEGG/Pfam)            ┆ 589    │
└──────────────────────────────────────────────────┴────────┘


Persist the per-protein master.

In [9]:
_p = TABLES / "avg_master_per_protein.parquet"
mast.rename({"homology_invisible": "sequence_similarity_invisible"}).write_parquet(_p)
print(f"avg_master_per_protein: {mast.height:,} x {mast.width}  ({_p.stat().st_size/1e6:.0f} MB)")
print(mast.group_by("function_level").agg(pl.len().alias("n")).sort("n", descending=True))

avg_master_per_protein: 541,199 x 48  (45 MB)


shape: (5, 2)
┌────────────────┬────────┐
│ function_level ┆ n      │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ annotate       ┆ 432092 │
│ specific       ┆ 45392  │
│ unassigned     ┆ 37957  │
│ L1             ┆ 20651  │
│ category       ┆ 5107   │
└────────────────┴────────┘


Per-protein category membership, long, from both sources.

Annotate rows come from the three curated category tables filtered to viral-confident proteins. Propagated rows come from specific labels expanded through the reference mapping to every (L1, L2) parent they belong to, which is how annotate-derived proteins are already counted. Labels assigned only to L1 depth carry a null L2 rather than an imputed one.

In [10]:
sp2l2 = pl.read_parquet(PROP / "fig_specific_to_L2.parquet")
_CATFILE = [("metabolic", "metabolic_gene_categories"),
            ("physiological", "physiology_gene_categories"),
            ("regulatory", "regulation_gene_categories")]
ann_cat = pl.concat([
    pl.read_parquet(anno(ds) / f"{fn}.parquet",
                    columns=["Protein", "Genome", "Function", "category_L1", "category_L2", "category_L3",
                             "Protein Viral Origin Confidence"])
      .filter(pl.col("Protein Viral Origin Confidence").is_in(VIRAL_OK))
      .with_columns(pl.lit(cat).alias("category"), pl.lit(ds).alias("ecosystem"),
                    pl.lit("annotate").alias("source"),
                    pl.lit(False).alias("homology_invisible"),
                    pl.lit(None, dtype=pl.String).alias("final_level"))
      .select(["Protein", "Genome", "ecosystem", "category", "category_L1", "category_L2",
               "category_L3", "Function", "source", "homology_invisible", "final_level"])
    for cat, fn in _CATFILE for ds in DATASETS])
# L3 is annotated for only a small minority of curated genes and is absent from the propagation
# mapping, so propagated labels inherit an L3 only where the same function name carries one in the
# curated tables. Everything else keeps a null L3 and falls back to L2 downstream.
_f2l3 = (ann_cat.filter(pl.col("category_L3").is_not_null())
         .select(["Function", "category_L3"]).unique(["Function"]))
print(f"Function -> L3 map: {_f2l3.height:,} functions")

assert set(mast["final_level"].unique()) == {"specific", "L1", "category", "unassigned"}, \
    "final_level vocabulary changed; recheck the specific and L1 gate below"
_pspec = (mast.filter(pl.col("final_level") == "specific")
          .join(sp2l2.select([pl.col("specific").alias("final_label"), "category_L1", "category_L2"]),
                on="final_label", how="inner")
          .join(_f2l3.rename({"Function": "final_label"}), on="final_label", how="left")
          .select(["Protein", "Genome", "ecosystem", pl.col("raw_category").alias("category"),
                   "category_L1", "category_L2", "category_L3",
                   pl.col("final_label").alias("Function"),
                   "homology_invisible", "final_level"])
          .with_columns(pl.lit("de-novo (propagated)").alias("source")))
_pL1 = (mast.filter((pl.col("final_level") == "L1") & pl.col("raw_L1").is_not_null())
        .select(["Protein", "Genome", "ecosystem", pl.col("raw_category").alias("category"),
                 pl.col("raw_L1").alias("category_L1"),
                 pl.lit(None, dtype=pl.String).alias("category_L2"),
                 pl.lit(None, dtype=pl.String).alias("category_L3"),
                 pl.col("final_label").alias("Function"), "homology_invisible", "final_level"])
        .with_columns(pl.lit("de-novo (propagated)").alias("source")))
pcat = pl.concat([ann_cat,
                  _pspec.select(ann_cat.columns), _pL1.select(ann_cat.columns)])
_p = TABLES / "avg_protein_categories.parquet"
pcat.rename({"homology_invisible": "sequence_similarity_invisible"}).write_parquet(_p)
print(f"avg_protein_categories: {pcat.height:,} rows  ({_p.stat().st_size/1e6:.0f} MB)")
print(pcat.group_by(["source", "category"]).agg(pl.len().alias("n")).sort(["source", "n"], descending=[False, True]))

Function -> L3 map: 127 functions


avg_protein_categories: 871,764 rows  (17 MB)
shape: (7, 3)
┌──────────────────────┬───────────────┬────────┐
│ source               ┆ category      ┆ n      │
│ ---                  ┆ ---           ┆ ---    │
│ str                  ┆ str           ┆ u32    │
╞══════════════════════╪═══════════════╪════════╡
│ annotate             ┆ regulatory    ┆ 242983 │
│ annotate             ┆ metabolic     ┆ 90934  │
│ annotate             ┆ physiological ┆ 58023  │
│ de-novo (propagated) ┆ regulatory    ┆ 310399 │
│ de-novo (propagated) ┆ metabolic     ┆ 93525  │
│ de-novo (propagated) ┆ physiological ┆ 75797  │
│ de-novo (propagated) ┆ null          ┆ 103    │
└──────────────────────┴───────────────┴────────┘


Genome universe, for denominators and grouping.

Every genome screened by `annotate` in either collection, with its biome, ecosystem subtype, and predicted host. Genome-level tests need the genomes that lack a given category as much as the ones that carry it, and that denominator is not recoverable from an AVG-only table.

In [11]:
guniv = (comb.select(["Genome", "dataset"]).unique()
         .rename({"dataset": "ecosystem"})
         .with_columns(pl.col("Genome").str.split("|").list.first().alias("uvig"))
         .join(iph.select(["uvig", "host_taxon"]), on="uvig", how="left")
         .join(uvm, on="uvig", how="left").join(src, on="taxon_oid", how="left")
         .with_columns(pl.when(pl.col("ecosystem_subtype").is_in(["Unclassified", "unclassified"]))
                         .then(None).otherwise(pl.col("ecosystem_subtype")).alias("ecosystem_subtype"),
                       pl.col("host_taxon").str.extract(r"p__([^;]+)").alias("host_phylum"))
         .select(["Genome", "ecosystem", "ecosystem_subtype",
                  pl.col("host_taxon").alias("host_taxonomy"), "host_phylum"]))
_p = TABLES / "avg_genome_universe.parquet"
guniv.write_parquet(_p)
print(f"avg_genome_universe: {guniv.height:,} genomes  ({_p.stat().st_size/1e6:.0f} MB)")
print(guniv.group_by("ecosystem").agg(pl.len().alias("genomes"),
                                      pl.col("ecosystem_subtype").n_unique().alias("subtypes"),
                                      pl.col("host_phylum").n_unique().alias("host_phyla")))

avg_genome_universe: 1,005,820 genomes  (16 MB)
shape: (2, 4)
┌───────────┬─────────┬──────────┬────────────┐
│ ecosystem ┆ genomes ┆ subtypes ┆ host_phyla │
│ ---       ┆ ---     ┆ ---      ┆ ---        │
│ str       ┆ u32     ┆ u32      ┆ u32        │
╞═══════════╪═════════╪══════════╪════════════╡
│ soil      ┆ 767786  ┆ 51       ┆ 97         │
│ gut       ┆ 238034  ┆ 8        ┆ 43         │
└───────────┴─────────┴──────────┴────────────┘


Embedding coordinates, cluster assignment, and neighbor mixing, per protein.

In [12]:
upts = ak.select(["Protein", "Genome", "ecosystem", "umap1", "umap2", "cluster", "category",
                  "function_label", "function_level", "function_source", "raw_L1", "host_taxonomy",
                  "frac_nn_same_ecosystem", "frac_nn_same_category"])
_p = TABLES / "avg_umap_points.parquet"
upts.write_parquet(_p)
print(f"avg_umap_points: {upts.height:,} proteins  ({_p.stat().st_size/1e6:.0f} MB)")
print(f"clusters (excluding noise): {upts.filter(pl.col('cluster') >= 0)['cluster'].n_unique()}")
print(f"noise points: {upts.filter(pl.col('cluster') < 0).height:,}")

avg_umap_points: 460,729 proteins  (15 MB)
clusters (excluding noise): 173
noise points: 74,744


## 6. Quality control

Checks that the emitted tables are internally consistent and joinable. These are assertions, not results.

Consistency and join integrity.

In [13]:
assert mast["Protein"].n_unique() == mast.height, "master must be one row per protein"
assert guniv["Genome"].n_unique() == guniv.height, "genome universe must be one row per genome"
_orphan = pcat.join(guniv.select("Genome"), on="Genome", how="anti").height
print(f"category rows whose genome is absent from the universe: {_orphan:,}")
assert _orphan == 0, "every categorized protein must belong to a screened genome"
_up = upts.join(mast.select("Protein"), on="Protein", how="anti").height
print(f"embedded proteins absent from the master: {_up:,}")
assert _up == 0, "every embedded protein must be in the master"
print(f"\nmaster proteins: {mast.height:,}")
print(f"category memberships: {pcat.height:,} over {pcat['Protein'].n_unique():,} distinct proteins")
print(f"genome universe: {guniv.height:,}")
print(f"embedded proteins: {upts.height:,}")
assert mast["label_source"].null_count() == 0, "label_source must survive the forwarding select"
assert mast["function_source"].null_count() == 0, "function_source must be populated for every protein"
_bad_none = mast.filter((pl.col("final_level") == "unassigned") != (pl.col("label_source") == "none")).height
print(f"proteins where final_level and label_source disagree about being unlabeled: {_bad_none:,}")
assert _bad_none == 0, "label_source none must coincide exactly with final_level unassigned"
print("\nfunction_source over the master:")
print(mast.group_by("function_source").agg(pl.len().alias("n")).sort("n", descending=True))
print("\nfunction_source over the sequence-similarity invisible AVGs:")
print(mast.filter(pl.col("homology_invisible"))
      .group_by("function_source").agg(pl.len().alias("n")).sort("n", descending=True))
print("\nassignment depth by tier, sequence-similarity invisible AVGs:")
print(mast.filter(pl.col("homology_invisible"))
      .group_by(["final_level", "label_source"]).agg(pl.len().alias("n"))
      .sort(["final_level", "n"], descending=[False, True]))
print("\nall row-level tables written; every summary and test is computed in soil_gut_avgs_figures.Rmd")

category rows whose genome is absent from the universe: 0
embedded proteins absent from the master: 0

master proteins: 541,199
category memberships: 871,764 over 474,977 distinct proteins
genome universe: 1,005,820
embedded proteins: 460,729
proteins where final_level and label_source disagree about being unlabeled: 0

function_source over the master:
shape: (4, 2)
┌─────────────────┬────────┐
│ function_source ┆ n      │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ annotate        ┆ 432092 │
│ embedding       ┆ 44606  │
│ none            ┆ 37957  │
│ cluster         ┆ 26544  │
└─────────────────┴────────┘

function_source over the sequence-similarity invisible AVGs:
shape: (3, 2)
┌─────────────────┬───────┐
│ function_source ┆ n     │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ embedding       ┆ 44606 │
│ none            ┆ 37957 │
│ cluster         ┆ 26544 │
└─────────────────┴───────┘

assignment d